# 03 · Agents and the LangGraph workflow (real Groq models)

```
START ─► supervisor ──Send (in parallel)──► specialist ─► synthesize ─► guardrail ─► END
          RoutePlan                         ReAct agent     FinalAnswer    drops invented
          (structured output)               + its tools     (structured)   citations
```

| Role | Groq model | Why |
|---|---|---|
| supervisor | `openai/gpt-oss-120b` | one small routing call |
| specialist | `openai/gpt-oss-20b` | tool calling, most tokens |
| synthesizer | `openai/gpt-oss-120b` | one structured answer |

The free tier allows 8,000 tokens per minute **per model**, so splitting the roles across two models doubles the budget.

**What you will learn:** each step on its own, then the whole graph, then the guardrail.

In [ ]:
# Setup: make the project importable and quiet the HTTP logs
import sys, logging, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("httpx").setLevel(logging.WARNING)

def show(obj, limit=1500):
    """Pretty-print a pydantic model, dict or list, cut to `limit` characters."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    if isinstance(obj, list):
        obj = [o.model_dump() if hasattr(o, "model_dump") else o for o in obj]
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n... (cut)")

In [ ]:
from pharma_agent.config import load_config
from pharma_agent.service import PharmaAgent

cfg = load_config(overrides={"store.path": ":memory:"})
agent = await PharmaAgent.create(cfg)      # real Groq models, tools in-process
for name, s in agent.specialists.items():
    print(f"{name:17s} tools={s.spec.tools}")

## The compiled graph (Mermaid text; paste into https://mermaid.live to draw it)

In [ ]:
print(agent.graph.get_graph().draw_mermaid())

## Step 1 · The supervisor alone: structured output (`RoutePlan`)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from pharma_agent.agents.prompts import SUPERVISOR
from pharma_agent.llm import build_llms
from pharma_agent.schemas import RoutePlan

llms = build_llms(cfg["llm"])
catalog = "\n".join(f"- {n}: {s.spec.description}" for n, s in agent.specialists.items())
question = "What are the main safety warnings for empagliflozin, and are there trials of it in chronic kidney disease?"

plan = await llms.supervisor.with_structured_output(RoutePlan).ainvoke(
    [SystemMessage(SUPERVISOR.format(catalog=catalog)), HumanMessage(question)])
show(plan)

## Step 2 · One specialist alone: a ReAct agent calling its tools

In [ ]:
text, evidence, calls = await agent.specialists["label_agent"].run("Find the boxed warning and key warnings for empagliflozin.")
print("Tool calls:", calls)
print("Evidence:", [e.id for e in evidence])
print("\nSummary:\n", text)

## Step 3 · The whole graph, and every intermediate result

In [ ]:
state = await agent.run_graph(question)
print("Plan:", [(t.agent, t.instruction) for t in state["plan"].tasks])
print("\nTool calls:")
for c in state["tool_calls"]:
    print("  ", c)
print("\nFindings:")
for f in state["findings"]:
    print(f"  [{f.agent}] {f.text[:300]}\n")

In [ ]:
show(state["answer"], 3000)
print("\nGuardrail:", state["guardrail"])

## Step 4 · The guardrail catches invented citations
LLMs sometimes cite ids that no tool returned. Here we take the real answer and add one made-up id to see
what the guardrail does: it removes the id, counts the point as unsupported and lowers confidence.

In [ ]:
from pharma_agent.agents import check_citations, unique_evidence
from pharma_agent.schemas import KeyPoint

tampered = state["answer"].model_copy(deep=True)
tampered.key_points.append(KeyPoint(text="A claim with a made-up source", citations=["pmid:00000000"]))
tampered.confidence = "high"
cleaned, report = check_citations(tampered, unique_evidence(state["evidence"]))
print(report)
print("Confidence after the check:", cleaned.confidence)

## Try it yourself
* Ask a literature question, e.g. *"What does research say about metformin in polycystic ovary syndrome?"*
* Change a specialist's `prompt` in `cfg["agents"]`, rebuild with `PharmaAgent.create(cfg)`, and compare.
* Next: **04_api_and_evaluation.ipynb**.